<a href="https://colab.research.google.com/github/pelineceburgun/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding A — ML Appendix, "What Predicts Health?" (Random Forest feature importance, p.27)

**The finding:** a Random Forest predicting `health_score` reports Average Position at 43% importance,
Impressions at 32%, Scroll Depth at 15%, and CTR at 8% — together 98% of the model's importance mass.

**Where does the label come from?** The paper's own "How to Read This Paper" page (p.5) defines
`health_score = Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts)`.
Three of the model's four top "predictors" — Position, Impressions, and CTR — are literally the
arithmetic components the label is built from, and Scroll Depth is the fourth. This is the leakage
skill's **Category 1 (label-derived features)** almost by definition: the label was computed FROM
these columns, and those columns are the features.

**My methodology question, respectfully:** the paper's caption already flags this ("the target itself
is partly constructed from some of these inputs, so importance is descriptive rather than causal"),
which is the right instinct — but the chart still ships with an implicit ranking (Position first,
then Impressions, then Scroll, then CTR) that a reader can misread as *which lever to pull first*.
Given the skill's own confession test — train once WITH the suspect feature, once WITHOUT, and watch
for a collapse — would it be worth publishing a second version of this chart restricted to inputs
that are **not** in the health-score formula (Content Age, Word Count, Days Visible, AI Sessions —
already the four features sitting at ~0% importance in the same chart)? That version would answer a
genuinely different, non-circular question: *of the things not already baked into the score, what
correlates with a strong page?* Right now those four features read as "unimportant," when what the
chart actually shows is "not part of the formula," which is a different claim.

### Finding B — ML Appendix, "What Predicts Growth?" (logistic regression, 71% holdout accuracy, p.29)

**The finding:** a logistic regression trained on the sampled active-content set gets 71% holdout
accuracy separating growing from declining pages, with Content Age, Days Since Update, and Days
Visible as the strongest signals.

**Does the validation design carry the claim?** Two concrete questions, in the spirit of the skill's
"attack your own model" checklist:

1. **Base rate.** Finding #1 (p.6), same up/down label family, reports 74.8K growing vs. 45.6K
   declining rows in the direct-comparison table — a ~62% growing base rate. If the 61.8K-row ML
   sample carries a similar split, 71% accuracy is only **~9 points of skill over the majority-class
   guess**, not 71 points. The 71% figure appears without a base rate next to it anywhere on that
   page, which the skill flags directly ("accuracy without its base rate next to it" is on the banned
   list).
2. **Split design.** The study spans 57 brands. The page doesn't say whether "holdout" means a random
   row split or a brand-grouped one. My own Week-5 model (Section 2 below) shows this isn't
   theoretical: moving the identical model from a random split to a client-grouped split cost
   **10–15 points of average precision** — the same portfolio, the same features, just an honest
   split. If FlyRank's 71% came from a random row split across 57 brands, part of that number is
   plausibly the model memorizing brand-level baselines rather than a transferable growth signal.

Framed constructively: printing the base rate next to the 71% and stating whether the holdout was
grouped by brand (or better, time-based, given the six months of trend data on p.17) would let a
reader judge how much of the 71% is real skill versus base rate and brand memorization — exactly the
kind of gap Section 2 makes visible on my own model.

In [1]:
# Grounding both methodology questions in numbers stated elsewhere in the paper itself
# (no external fetch needed -- these are the paper's own reported figures, p.6 and p.27/29)

health_score_formula_share = {"avg_position": 43, "impressions": 32, "scroll_depth": 15, "ctr": 8}
print("Finding A -- share of RF importance sitting on health-score formula components:")
print(f"  {sum(health_score_formula_share.values())}% of importance on the 4 features that ARE the label's formula")
print(f"  (formula: health_score = impressions(30) + position(30) + ctr(20) + scroll_depth(20), p.5)")

growing, declining = 74_800, 45_600  # Finding #1, p.6 (74.8K vs 45.6K)
base_rate = growing / (growing + declining)
reported_accuracy = 0.71
print(f"\nFinding B -- majority-class base rate implied by Finding #1's up/down counts: {base_rate:.3f}")
print(f"  reported holdout accuracy: {reported_accuracy:.3f}")
print(f"  skill over base rate: {reported_accuracy - base_rate:+.3f} (~{(reported_accuracy-base_rate)*100:.0f} points, not {reported_accuracy*100:.0f})")

Finding A -- share of RF importance sitting on health-score formula components:
  98% of importance on the 4 features that ARE the label's formula
  (formula: health_score = impressions(30) + position(30) + ctr(20) + scroll_depth(20), p.5)

Finding B -- majority-class base rate implied by Finding #1's up/down counts: 0.621
  reported holdout accuracy: 0.710
  skill over base rate: +0.089 (~9 points, not 71)


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 model already used a client-grouped split (see `w05_model.ipynb`, Section 2), for exactly
the reason Finding B above raises: `client_id` must define the split, or the model can learn
per-client baselines instead of a transferable "is this page losing ground" pattern. What I hadn't
done yet is show the **before** — the same data, same features, same three models, scored on a naive
random row split — next to the honest **after**, so the gap is visible rather than assumed.

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

df = pd.read_csv(
    "https://raw.githubusercontent.com/pelineceburgun/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
)

# Label (same as W05/W02: leakage-safe, trend_direction/trend_pct are the label family, never features)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Same leakage-safe feature set as w05_model.ipynb -- impressions_last_30d / impressions_prev_30d
# excluded here on purpose. Section 3 below re-runs the ablation that proves why.
numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "clicks_last_30d", "sessions_last_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
categorical_features = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

num = df[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat = df[categorical_features].fillna("unknown").astype(str)
cat_enc = pd.get_dummies(cat, prefix=categorical_features, dtype=float)
X = pd.concat([num.reset_index(drop=True), cat_enc.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].reset_index(drop=True)

def precision_at_k(y_true, scores, k=50):
    d = pd.DataFrame({"y": np.asarray(y_true), "score": np.asarray(scores)})
    top = d.sort_values("score", ascending=False).head(min(k, len(d)))
    return float(top["y"].mean())

def make_models():
    return {
        "logistic_regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
        ]),
        "decision_tree": DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=RANDOM_STATE),
        "random_forest": RandomForestClassifier(
            n_estimators=300, max_depth=8, min_samples_leaf=20,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
        ),
    }

def fit_score(Xmat, train_idx, test_idx):
    Xt, Xte = Xmat.iloc[train_idx], Xmat.iloc[test_idx]
    yt, yte = y.iloc[train_idx], y.iloc[test_idx]
    rows, fitted = {}, {}
    for name, model in make_models().items():
        model.fit(Xt, yt)
        fitted[name] = model
        proba = model.predict_proba(Xte)[:, 1]
        rows[name] = {
            "precision_at_50": precision_at_k(yte, proba, 50),
            "avg_precision": average_precision_score(yte, proba),
            "roc_auc": roc_auc_score(yte, proba),
        }
    rows["base_rate (test rows)"] = {
        "precision_at_50": float(yte.mean()), "avg_precision": float(yte.mean()), "roc_auc": 0.5
    }
    return pd.DataFrame(rows).T.round(3), fitted

# ---- BEFORE: naive random row split (ignores that rows cluster by client_id) ----
rand_train_idx, rand_test_idx = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
before_df, _ = fit_score(X, rand_train_idx, rand_test_idx)

# ---- AFTER: grouped split by client_id (identical logic to w05_model.ipynb Section 2) ----
client_series = df["client_id"].astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(unique_clients)
n_test_clients = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test_clients])
test_mask = client_series.isin(test_clients).to_numpy()
grp_train_idx, grp_test_idx = np.where(~test_mask)[0], np.where(test_mask)[0]
print(f"grouped split: {len(unique_clients)} clients total, {n_test_clients} held out, "
      f"{len(grp_train_idx)} train rows, {len(grp_test_idx)} test rows")
print(f"label rate -- random-split test: {y.iloc[rand_test_idx].mean():.3f}, "
      f"grouped-split test: {y.iloc[grp_test_idx].mean():.3f}")

after_df, fitted_after = fit_score(X, grp_train_idx, grp_test_idx)

comparison = pd.concat(
    {"before: random row split": before_df, "after: client-grouped split": after_df},
    names=["split", "model"],
)
comparison

grouped split: 32 clients total, 6 held out, 27675 train rows, 2325 test rows
label rate -- random-split test: 0.542, grouped-split test: 0.391


precision_at_50  \
split                       model                                    
before: random row split    logistic_regression              0.880   
                            decision_tree                    0.880   
                            random_forest                    0.900   
                            base_rate (test rows)            0.542   
after: client-grouped split logistic_regression              0.640   
                            decision_tree                    0.540   
                            random_forest                    0.860   
                            base_rate (test rows)            0.391   

                                                   avg_precision  roc_auc  
split                       model                                          
before: random row split    logistic_regression            0.735    0.725  
                            decision_tree                  0.706    0.723  
                            random_forest                  0.772    0.764  
                            base_rate (test rows)          0.542    0.500  
after: client-grouped split logistic_regression            0.614    0.740  
                            decision_tree                  0.601    0.759  
                            random_forest                  0.670    0.775  
                            base_rate (test rows)          0.391    0.500

In [3]:
gap = (before_df["avg_precision"] - after_df["avg_precision"]).round(3)
print("gap in average precision (before minus after, honest split):")
print(gap)

gap in average precision (before minus after, honest split):
logistic_regression      0.121
decision_tree            0.105
random_forest            0.102
base_rate (test rows)    0.151
Name: avg_precision, dtype: float64


**Reading the gap.** Every model looks meaningfully better under the random split than under the
client-grouped one — average precision drops 0.10-0.12 for the three learned models once client
membership is honestly held out, and even the **base rate itself** shifts (0.542 on the random split's
test rows vs. 0.391 on the grouped split's held-out clients), because a random split mixes in
easy-and-hard clients evenly while a client holdout can land on a batch of clients with a
structurally different decline rate. Random Forest still wins on both splits and still clears its
honest base rate by a wide margin (0.670 vs. 0.391), so the qualitative conclusion from Week 5
survives — but the *size* of the win was overstated under the random split, exactly the pattern the
leakage-and-validation skill predicts ("a random split lets the model memorize the group and fake
skill"). This is the same category of gap I raised about Finding B's 71% figure in Section 1: I can't
know how much of that number is real without knowing which split produced it, and here, on my own
data, the honest number is 10-12 points lower than the convenient one.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Running the attack checklist from `hunting-leakage-and-validating/SKILL.md` against the exact
feature set used above (Section 2, "after").

In [4]:
# --- Checklist item 1: no label-derived / sibling columns in the features ---
label_family = {"trend_direction", "trend_pct"}
assert label_family.isdisjoint(X.columns), "label-family column leaked into features"
print("[pass] trend_direction / trend_pct not in the feature matrix")

# --- Checklist item 2: IDs never used as features (grouping/joining only) ---
id_cols = {"content_id", "client_id"}
assert id_cols.isdisjoint(X.columns), "an ID column leaked into features"
print("[pass] content_id / client_id not in the feature matrix (client_id used only to build the split)")

# --- Checklist item 3: the suspect confession test ---
# impressions_last_30d / impressions_prev_30d were excluded from the W05 feature set after they
# turned out to BE the label's formula. Re-running the skill's verification step here: add them
# back, watch the score jump toward 1.0, then confirm they stay out of the real feature set.
suspects = ["impressions_last_30d", "impressions_prev_30d"]
num_suspect = df[suspects].apply(pd.to_numeric, errors="coerce").fillna(0)
X_with_suspects = pd.concat([X, num_suspect.reset_index(drop=True)], axis=1)

without_df, _ = fit_score(X, grp_train_idx, grp_test_idx)
with_df, _ = fit_score(X_with_suspects, grp_train_idx, grp_test_idx)
print("\nWITHOUT suspects (this notebook's real feature set):")
print(without_df[["avg_precision", "roc_auc"]])
print("\nWITH impressions_last_30d/prev_30d added back:")
print(with_df[["avg_precision", "roc_auc"]])

formula = (df["impressions_last_30d"] - df["impressions_prev_30d"]) / df["impressions_prev_30d"].replace(0, np.nan) * 100
print(f"\ntrend_pct correlation with (last30-prev30)/prev30*100: {df['trend_pct'].corr(formula):.10f}")

# --- Checklist item 4: no other near-perfect correlation hiding in the final numeric features ---
corrs = num.apply(lambda c: np.corrcoef(c, y)[0, 1]).sort_values(key=lambda s: s.abs(), ascending=False)
print("\ntop absolute correlations, final numeric features vs. label (none near 1.0):")
print(corrs.head(6).round(3))

# --- Checklist item 5: base rate printed next to the metric (done above and again here) ---
print(f"\nbase rate on held-out clients: {y.iloc[grp_test_idx].mean():.3f}")

# --- Checklist item 6: top feature importance sanity check (permutation importance, honest split) ---
best_model = fitted_after["random_forest"]
perm = permutation_importance(
    best_model, X.iloc[grp_test_idx], y.iloc[grp_test_idx],
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1, scoring="average_precision",
)
importance = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
print("\ntop 5 features by permutation importance (drop in avg precision when shuffled):")
print(importance.head(5).round(4))

[pass] trend_direction / trend_pct not in the feature matrix
[pass] content_id / client_id not in the feature matrix (client_id used only to build the split)

WITHOUT suspects (this notebook's real feature set):
                       avg_precision  roc_auc
logistic_regression            0.614    0.740
decision_tree                  0.601    0.759
random_forest                  0.670    0.775
base_rate (test rows)          0.391    0.500

WITH impressions_last_30d/prev_30d added back:
                       avg_precision  roc_auc
logistic_regression            0.821    0.849
decision_tree                  0.792    0.896
random_forest                  0.901    0.936
base_rate (test rows)          0.391    0.500

trend_pct correlation with (last30-prev30)/prev30*100: 0.9999999984

top absolute correlations, final numeric features vs. label (none near 1.0):
days_with_impressions     0.190
content_age_days         -0.164
word_count                0.119
char_count                0.108
days_

**Reading the audit.** The confession test confirms the Week-5 exclusion decision: adding
`impressions_last_30d`/`impressions_prev_30d` back in lifts Random Forest average precision from
0.670 to about 0.90 and ROC AUC from 0.775 to about 0.94 — the label-in-disguise jump the skill
describes, driven by `trend_pct` correlating essentially 1.0 with the exact arithmetic of those two
columns. With them excluded, the strongest remaining correlation between any single numeric feature
and the label is `days_with_impressions` at about 0.19 — nowhere near the "suspiciously perfect"
range, and it lines up with the top permutation-importance feature too, which is the sane result: a
feature that's genuinely informative, not one that's secretly the answer key. No content/client IDs
and no `trend_direction`/`trend_pct` sit in the feature matrix, and the split is grouped by
`client_id` rather than random. Checklist: clean.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**As I was first tempted to write it (Week 5 draft, before I caught myself):**

> "Random Forest predicts which pages are about to decline, so the model tells us exactly which
> content to refresh first."

That's a causal, near-certain claim from a cross-sectional model on one held-out slice of one
portfolio — banned language per `writing-honest-claims/SKILL.md` ("predicts... exactly" implies a
guarantee the evidence can't carry, and it skips the base rate and the error pattern entirely).

**Rewritten to match the claim ladder** (evidence: a validated model that ranks out-of-sample,
grouped by client — one rung below "controlled experiment," so decision-support language, not
causal language):

In [5]:
best_ap = float(after_df.loc["random_forest", "avg_precision"])
base = float(after_df.loc["base_rate (test rows)", "avg_precision"])
print(
    f"On content held out by client (clients the model never trained on), Random Forest ranks pages "
    f"by measured decline risk with average precision {best_ap:.3f} against a base rate of {base:.3f} "
    f"on this same held-out slice -- a real, directional improvement over guessing, not a guarantee "
    f"for any single page. In the Week-5 error review, 7 of the top-50 flagged pages were not "
    f"actually declining, and every one of those misses was a page updated in the last 20 days and "
    f"currently trending up or stable -- so the honest use is decision-support: a 'worth a look' "
    f"queue for reviewers, cross-checked against recent update date, not an automatic 'confirmed "
    f"declining' verdict."
)

On content held out by client (clients the model never trained on), Random Forest ranks pages by measured decline risk with average precision 0.670 against a base rate of 0.391 on this same held-out slice -- a real, directional improvement over guessing, not a guarantee for any single page. In the Week-5 error review, 7 of the top-50 flagged pages were not actually declining, and every one of those misses was a page updated in the last 20 days and currently trending up or stable -- so the honest use is decision-support: a 'worth a look' queue for reviewers, cross-checked against recent update date, not an automatic 'confirmed declining' verdict.


| Evidence I actually have | Word choice |
|---|---|
| A validated model, ranking out-of-sample, grouped by client | "ranks... at average precision of..." (not "predicts exactly") |
| One cross-sectional snapshot, no intervention | never "will decline" / "causes decline" |
| A known failure pattern (recently-updated pages false-flagged) | named explicitly, not smoothed over |

This mirrors the same question I asked of FlyRank's Finding B in Section 1 — a bare accuracy or
precision number reads stronger than it is until the base rate and the known failure mode sit right
next to it in the same sentence.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.